In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, when

In [2]:
BUCKET_NAME = "healthcare-tungpd20"

# Read all CSV files inside landing/claims/
CLAIMS_BUCKET_PATH = f"gs://{BUCKET_NAME}/landing/claims/*.csv"

BQ_PROJECT = "project-8bf7907a-2104-49b0-99f"
BQ_TABLE = f"{BQ_PROJECT}.bronze_dataset.claims"

TEMP_GCS_BUCKET = f"gs://{BUCKET_NAME}/temp/"

In [3]:
claims_df = spark.read.csv(CLAIMS_BUCKET_PATH, header = True)

In [7]:
claims_df = (claims_df
                .withColumn("datasource", 
                              when(input_file_name().contains("hospital2"), "hosb")
                             .when(input_file_name().contains("hospital1"), "hosa").otherwise("None")))

In [9]:
claims_df.printSchema()

root
 |-- claimid: string (nullable = true)
 |-- transactionid: string (nullable = true)
 |-- patientid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- providerid: string (nullable = true)
 |-- deptid: string (nullable = true)
 |-- servicedate: string (nullable = true)
 |-- claimdate: string (nullable = true)
 |-- payorid: string (nullable = true)
 |-- claimamount: string (nullable = true)
 |-- paidamount: string (nullable = true)
 |-- claimstatus: string (nullable = true)
 |-- payortype: string (nullable = true)
 |-- deductible: string (nullable = true)
 |-- coinsurance: string (nullable = true)
 |-- copay: string (nullable = true)
 |-- insertdate: string (nullable = true)
 |-- modifieddate: string (nullable = true)
 |-- datasource: string (nullable = false)



In [10]:
claims_df = claims_df.dropDuplicates()

In [11]:
(claims_df.write
            .format("bigquery")
            .option("table", BQ_TABLE)
            .option("temporaryGcsBucket", TEMP_GCS_BUCKET)
            .mode("overwrite")
            .save())